##Silver Layer Injestion

In [0]:
%sql
Use catalog olist_ecommerce_project;


### Customers Table Data Manipulation and Cleaning

In [0]:
df_customers_bronze = spark.table("olist_ecommerce_project.bronze.brz_customers")

# Basic profiling
print("Total rows:", df_customers_bronze.count())

Unique customer_id Total Count

In [0]:
print("Distinct customer_id:", df_customers_bronze.select("customer_id").distinct().count())

Unique cutomer_unique_id Total Count

In [0]:
print("Distinct customer_unique_id:", df_customers_bronze.select("customer_unique_id").distinct().count())

Checking the column data types of customer table

In [0]:
df_customers_bronze.printSchema()

Checking Null Values

In [0]:
# Null check across all columns
from pyspark.sql.functions import col, sum as spark_sum

df_customers_bronze.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) for c in df_customers_bronze.columns
]).show()

Checking the City and State columns

In [0]:
df_customers_bronze.select("customer_city").distinct().orderBy("customer_city").show(30, truncate=False)

In [0]:
df_customers_bronze.select("customer_state").distinct().orderBy("customer_state").show(30, truncate=False)

Injesting into Silver Table

In [0]:
df_customers_silver = df_customers_bronze

# No cleaning needed - already unique, no nulls, consistent casing
# We simply promote it to Silver, dropping Bronze-only audit columns 
# and replacing with Silver-specific ones if desired

df_customers_silver = df_customers_silver.drop("_source_file")  # keep ingestion_timestamp as lineage info, drop source_file since it's no longer relevant past Bronze

(
    df_customers_silver.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable("olist_ecommerce_project.silver.slv_customers")
)